# 🎙️ GSoC 2026 — AutoEIT: Test I — Audio Transcription Pipeline
**Applicant:** Jb Anmol | **Organization:** HumanAI Foundation | **Project:** AutoEIT

---

## 📌 1. Objective & Scientific Motivation

The objective of Test I is to generate **research-grade transcriptions** for four sample audio files containing non-native Spanish Elicited Imitation Task (EIT) recordings — 30 target sentences per participant, 120 utterances total.

A naive approach would be to simply pass the audio through a Speech-to-Text model and call it done. However, this introduces a **critical scientific flaw**: state-of-the-art ASR models are trained to maximize linguistic correctness, which causes them to aggressively auto-correct learner phonetic errors into grammatically perfect sentences. In an EIT research context, this is not a transcription improvement — it is **data falsification**. The moment Whisper silently converts a learner's actual utterance *"coltarme"* into the standard *"cortarme"*, the downstream scoring rubric is no longer measuring the learner's proficiency — it is measuring Whisper's correction ability.

The transcription pipeline must therefore preserve **exact phonetic production**, including mispronunciations, false starts, stutters, and hesitations, as these are precisely the signals the EIT scoring rubric is designed to evaluate.

---

## 🏗️ 2. Pipeline Architecture

To solve this, I designed a **Hybrid AI-Human Pipeline** that uses ASR for speed and structural scaffolding, while reserving phonetic accuracy decisions for a human auditor.

In [1]:
# @title Pipeline Architecture Diagram
from IPython.display import HTML
HTML("""
<div style="font-family: monospace; font-size: 13px; margin: 20px 0;">
<div style="display: flex; align-items: center; gap: 0;">

  <div style="background:#1e3a5f; color:white; border:2px solid #4a90d9; border-radius:8px; padding:12px 16px; text-align:center; min-width:140px;">
    🎵 <b>Raw Audio</b><br><span style="font-size:11px; color:#a8c8f0;">4 × .mp3 files</span>
  </div>

  <div style="color:#4a90d9; font-size:22px; padding:0 6px;">→</div>

  <div style="background:#1e3a5f; color:white; border:2px solid #4a90d9; border-radius:8px; padding:12px 16px; text-align:center; min-width:140px;">
    🤖 <b>faster-whisper</b><br><span style="font-size:11px; color:#a8c8f0;">large-v3 + dynamic offsets</span>
  </div>

  <div style="color:#4a90d9; font-size:22px; padding:0 6px;">→</div>

  <div style="background:#1e3a5f; color:white; border:2px solid #4a90d9; border-radius:8px; padding:12px 16px; text-align:center; min-width:140px;">
    📄 <b>Draft Audit CSV</b><br><span style="font-size:11px; color:#a8c8f0;">30 items + UNASSIGNED rows</span>
  </div>

  <div style="color:#e8832a; font-size:22px; padding:0 6px;">→</div>

  <div style="background:#7b3f00; color:white; border:2px solid #e8832a; border-radius:8px; padding:12px 16px; text-align:center; min-width:140px;">
    👁️ <b>Human Audit</b><br><span style="font-size:11px; color:#ffd0a0;">120 items reviewed vs. raw audio</span>
  </div>

  <div style="color:#2ecc71; font-size:22px; padding:0 6px;">→</div>

  <div style="background:#1a4731; color:white; border:2px solid #2ecc71; border-radius:8px; padding:12px 16px; text-align:center; min-width:140px;">
    ✅ <b>Audited CSV</b><br><span style="font-size:11px; color:#a8f0c6;">FINAL_AUDIT_TRANSCRIPTION filled</span>
  </div>

  <div style="color:#2ecc71; font-size:22px; padding:0 6px;">→</div>

  <div style="background:#1a4731; color:white; border:2px solid #2ecc71; border-radius:8px; padding:12px 16px; text-align:center; min-width:140px;">
    📊 <b>Excel Write-Back</b><br><span style="font-size:11px; color:#a8f0c6;">openpyxl cell-level injection</span>
  </div>

</div>

<div style="display:flex; margin-top:10px; gap:20px; font-size:12px;">
  <span style="color:#4a90d9;">■ AI Processing</span>
  <span style="color:#e8832a;">■ Human-in-the-Loop</span>
  <span style="color:#2ecc71;">■ Output</span>
</div>
</div>
""")

# 📑 Table of Contents
1. Objective & Scientific Motivation
2. Pipeline Architecture
3. Challenges & Engineered Solutions
4. Implementation & Execution Logs
5. Output Quality & Evaluation
6. Final Deliverables
7. Reflections & Limitations

**Pipeline stages:**
1. **ASR Processing** — `faster-whisper` (large-v3) generates word-level timestamps and draft transcriptions for each audio file, with dynamic offsets to skip instructional audio.
2. **CSV Isolation** — A segmentation script maps ASR output to the 30 expected target items per participant and exports isolated audit CSVs, flagging any extra ASR rows as `UNASSIGNED_N`.
3. **Human-in-the-Loop Audit** — Each draft transcription was manually reviewed against the raw audio. The `FINAL_AUDIT_TRANSCRIPTION` column was filled using a standardized phonetic notation system.
4. **Automated Write-Back** — A final injection script reads the audited CSVs and writes verified transcriptions into the master Excel workbook, preserving all original cell formatting, formulas, and structure.

---

## ⚠️ 3. Challenges & Engineered Solutions

### Challenge A — The "Over-Smart" ASR Problem

**Issue:** Whisper large-v3 consistently auto-corrected learner production errors. Left unedited, these silent corrections would **artificially inflate learner scores** in Test II, rendering the rubric meaningless.

| Learner's Actual Utterance | Whisper's "Correction" | Error Type Erased |
|:---|:---|:---|
| *coltarme* | *cortarme* | Phonological substitution |
| *entla mesa* | *en la mesa* | Consonant liaison |
| *purperro* | *por el perro* | Vowel reduction |
| *keva* | *llueva* | Phonemic substitution |
| *Tuvo que sepa* | *Dudo que sepa* | Lexical hallucination |

**Solution:** The isolated CSV audit workflow enforced a strict separation between ASR output (`draft_asr_text`) and final transcription (`FINAL_AUDIT_TRANSCRIPTION`). A standardized phonetic notation system was applied consistently across all 120 items:

| Tag | Meaning | Example |
|:---|:---|:---|
| `[word-]` | False start / self-correction | `[d-]de personas` |
| ` ... ` | Vocalized pause / hesitation | `Me gustaría ... pronto` |
| Literal phonetic spelling | Exact phonetic shape as heard | `purperro`, `keva`, `entla` |
| `[unintelligible]` | Truly unintelligible segment | `[unintelligible]` |

---

### Challenge B — Irregular Audio Structure & Large Timestamp Offsets

**Issue:** Each audio file contained English instructional audio and practice sentences before the Spanish test began. File `038012_EIT-2A.mp3` had approximately **740 seconds (~12.3 minutes)** of instructional content before the first target sentence, confirmed by the first item's ASR timestamp of `740.22s`.

**Solution:** Rather than manually clipping audio files (which would alter timestamps and risk data loss), I implemented **dynamic offset parameters** directly into the Whisper processing call:

| Participant | Offset Applied | Reason |
|:---|:---|:---|
| 38010-2A, 38011-1A, 38015-1A | `150s` | Standard instructional lead-in |
| 38012-2A | `720s` | Extended ~12 min instructional block |

This preserved original timestamps for traceability, saved significant compute time, and prevented Whisper from hallucinating over long stretches of silence or English speech.

---

### Challenge C — ASR Hallucinations & Trailing Junk Rows

**Issue:** Because EIT participants repeat sentences with deliberate pauses between items, Whisper occasionally transcribed background noise, sighs, or end-of-session English speech as additional Spanish fragments.

| Participant | UNASSIGNED Rows Generated |
|:---|:---|
| 38010-2A | 19 |
| 38011-1A | 8 |
| 38012-2A | 1 |
| 38015-1A | 0 |

**Solution:** The segmentation script strictly maps output based on an expected item index of 1–30. All rows beyond index 30 are automatically flagged as `UNASSIGNED_N` with `notes = "Extra segment picked up by ASR"` and excluded from write-back. The master Excel receives exactly 30 clean, audited transcriptions per participant.

---

### Challenge D — Excel Format Preservation During Write-Back

**Issue:** The master workbook uses merged cells, conditional formatting, and fixed column structures. A naive `pandas` write (`df.to_excel()`) would silently destroy the entire formatting, breaking the research template.

**Solution:** I used `openpyxl` in read/write mode, targeting **only** the specific cells in the `FINAL_AUDIT_TRANSCRIPTION` column by row index. No other cells, styles, or formulas are touched. A timestamped safety backup (`.bak`) is created before every write operation.


---

## 🔧 5. Implementation & Execution Logs

The following cells document the complete pipeline execution on **Google Colab (Tesla T4 GPU)**. All outputs are preserved from the original run. Source code for each script is available in the `src/` directory of this repository.

### 5.1 Environment Setup
Project files are loaded from Google Drive, dependencies are installed, and the GPU runtime is verified.

In [2]:
from google.colab import drive
import os

# Mount Google Drive to access project files
drive.mount('/content/drive')

# Copy project zip from Drive to local environment
!cp "/content/drive/MyDrive/Gsoc_2026_humanAi/autoEIT_upload.zip" "/content/project.zip"

# Unzip the project files
!unzip -q /content/project.zip -d /content/

# Change directory to the unzipped project folder
%cd /content/autoEIT-audio-transcription-jbanmol9

# List directory contents
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/autoEIT-audio-transcription-jbanmol9
config	notebooks  pyproject.toml  requirements-colab.txt  uv.lock
data	outputs    README.md	   requirements.txt
docs	plan.md    reports	   src


In [3]:
# Install necessary Python packages for ASR and Excel handling
!pip install -q faster-whisper pandas openpyxl PyYAML

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 86.5 MB/s eta 0:00:00


In [4]:
import faster_whisper
import pandas as pd
import openpyxl
import yaml

print("OK: all dependencies import correctly")

OK: all dependencies import correctly


In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — go to Runtime > Change runtime type")

CUDA available: True
GPU: Tesla T4


In [6]:
# Verify config loads correctly
import yaml
with open("config/test1_metadata.yaml") as f:
    cfg = yaml.safe_load(f)
for entry in cfg["files"]:
    print(entry["participant_id"], "→", entry["audio_filename"], "| offset:", entry["start_offset_sec"], "s")

38010-2A → 038010_EIT-2A.mp3 | offset: 150 s
38011-1A → 038011_EIT-1A.mp3 | offset: 150 s
38012-2A → 038012_EIT-2A.mp3 | offset: 720 s
38015-1A → 038015_EIT-1A.mp3 | offset: 150 s


### 5.2 Batch ASR Processing

All four participant audio files are processed using `faster-whisper` (large-v3) with per-file dynamic `start_offset_sec` values. The model skips instructional English audio and begins transcription only after the configured offset.

> **Note:** A `--only-id` flag is available for single-file pilot testing. The cell below runs the full batch.

In [7]:
!python src/01_run_whisper.py

Initializing ASR Pipeline...
Files to be processed:
  - 38010-2A (038010_EIT-2A.mp3)
  - 38011-1A (038011_EIT-1A.mp3)
  - 38012-2A (038012_EIT-2A.mp3)
  - 38015-1A (038015_EIT-1A.mp3)
Loading faster-whisper model (large-v3)...

Processing 38010-2A: 038010_EIT-2A.mp3 (Skipping first 150s)
Detected language 'es' with probability 1.00
Saved outputs for 38010-2A

Processing 38011-1A: 038011_EIT-1A.mp3 (Skipping first 150s)
Detected language 'es' with probability 1.00
Saved outputs for 38011-1A

Processing 38012-2A: 038012_EIT-2A.mp3 (Skipping first 720s)
Detected language 'es' with probability 1.00
Saved outputs for 38012-2A

Processing 38015-1A: 038015_EIT-1A.mp3 (Skipping first 150s)
Detected language 'es' with probability 1.00
Saved outputs for 38015-1A


### 5.3 Draft Audit CSV Generation

Raw ASR segments are chronologically paired with the 30 expected target stimuli from the workbook. Segments beyond item 30 are automatically flagged as `UNASSIGNED_N` for human review.

In [8]:
!python src/02_prepare_audit_csv.py

Preparing draft review CSV files...
Generated outputs/review_csv/38010-2A_audit.csv with 49 rows for 38010-2A.
Generated outputs/review_csv/38011-1A_audit.csv with 38 rows for 38011-1A.
Generated outputs/review_csv/38012-2A_audit.csv with 31 rows for 38012-2A.
Generated outputs/review_csv/38015-1A_audit.csv with 30 rows for 38015-1A.


### 5.4 CSV Export for Offline Audit

The draft audit CSVs are packaged and downloaded for offline review against the raw audio waveform.

In [9]:
!zip -r batch_csvs.zip outputs/review_csv/
from google.colab import files
files.download('batch_csvs.zip')

  adding: outputs/review_csv/ (stored 0%)
  adding: outputs/review_csv/38012-2A_audit.csv (deflated 57%)
  adding: outputs/review_csv/38011-1A_audit.csv (deflated 66%)
  adding: outputs/review_csv/38010-2A_audit.csv (deflated 70%)
  adding: outputs/review_csv/38015-1A_audit.csv (deflated 61%)


### 5.5 🛑 Offline Manual Audit Phase

The draft audit CSVs were reviewed offline against the raw audio using **Audacity**. Each of the 120 items was individually verified:

1. **Tone Detection** — Identified the sharp tone marking each item boundary
2. **ASR Comparison** — Compared the learner's actual utterance against the Whisper draft
3. **Error Correction** — Restored every phonological error, false start, and hesitation that Whisper had silently auto-corrected (see Challenge A)
4. **Column Population** — Filled `FINAL_AUDIT_TRANSCRIPTION` using the standardized phonetic notation system
5. **Row Cleanup** — Removed all `UNASSIGNED_N` drift rows; verified exactly 30 items (1–30) per participant

The audited CSVs were then uploaded back to the Colab environment for write-back.

> ⚠️ This step is **intentionally manual**. Automating it would reintroduce the exact auto-correction bias the pipeline is designed to prevent.

### 5.6 Verified Excel Write-Back

The injection script performs strict pre-flight validation before modifying the master workbook:
- Creates a timestamped `.bak` safety backup
- Verifies exactly 30 rows with `item_number` 1–30 per participant
- Rejects any blank `FINAL_AUDIT_TRANSCRIPTION` cells
- Uses `openpyxl` cell-level injection, preserving all original formatting

In [10]:
# Write transcribed text back into the master Excel workbook
!python src/03_populate_excel.py

# Download the updated Excel file
from google.colab import files
files.download('data/working/AutoEIT Sample Audio for Transcribing_WORKING.xlsx')

Created safety backup at data/working/AutoEIT Sample Audio for Transcribing_WORKING_20260331_040922.bak

--- SAFETY CHECK: Excel Write-Back Initialization ---
Target Workbook: data/working/AutoEIT Sample Audio for Transcribing_WORKING.xlsx
Expected Audit CSV Files:
  - 38010-2A_audit.csv [FOUND]
  - 38011-1A_audit.csv [FOUND]
  - 38012-2A_audit.csv [FOUND]
  - 38015-1A_audit.csv [FOUND]
---------------------------------------------------


Evaluating final CSV for 38010-2A...
Successfully injected 30 transcripts into sheet '38010-2A'.

Evaluating final CSV for 38011-1A...
Successfully injected 30 transcripts into sheet '38011-1A'.

Evaluating final CSV for 38012-2A...
Successfully injected 30 transcripts into sheet '38012-2A'.

Evaluating final CSV for 38015-1A...
Successfully injected 30 transcripts into sheet '38015-1A'.

✅ Workbook successfully updated and verified. Formats preserved.



---

## 📊 4. Output Quality & Evaluation

The final output was evaluated through three lenses:

**1. Coverage**
All 4 participants × 30 items = **120 transcriptions** successfully populated. Zero missing values in `FINAL_AUDIT_TRANSCRIPTION` for items 1–30 across all four sheets.

**2. ASR Divergence Analysis**
Draft ASR output was compared against final audited transcriptions across all 120 items. The most common divergence patterns identified:
- **Phonological substitution** — Whisper mapped learner non-target sounds to the nearest standard phoneme
- **Morphosyntactic reconstruction** — Whisper reassembled fragmented learner output into complete, grammatical sentences
- **Lexical hallucination** — Whisper inserted words entirely absent from the learner's actual production
- **Cross-item bleed** — Missing sentence boundaries caused Whisper to merge adjacent items into one ASR segment

**3. Phonetic Fidelity**
Final transcriptions were cross-checked against the raw audio for every item where ASR output diverged from the audited version. Transcriptions preserve disfluencies, false starts, and non-target phonetic forms — ensuring the scoring rubric in Test II is applied to the **learner's actual utterance**, not an ASR-corrected proxy.

---

## 📁 5. Final Deliverable

| File | Description |
|:---|:---|
| `data/working/AutoEIT Sample Audio for Transcribing_WORKING.xlsx` | Master workbook — 120 audited transcriptions injected |
| `data/audit_csvs/38010-2A_audit.csv` | Participant 38010-2A (49 rows: 30 items + 19 UNASSIGNED) |
| `data/audit_csvs/38011-1A_audit.csv` | Participant 38011-1A (38 rows: 30 items + 8 UNASSIGNED) |
| `data/audit_csvs/38012-2A_audit.csv` | Participant 38012-2A (31 rows: 30 items + 1 UNASSIGNED) |
| `data/audit_csvs/38015-1A_audit.csv` | Participant 38015-1A (30 rows: 30 items, no UNASSIGNED) |

> ✅ All transcriptions are research-grade and ready for automated scoring in **Test II →**

---

---

## 🏁 7. Reflections & Limitations

**1. ASR as Draft, Not Ground Truth** — Whisper large-v3 generates useful scaffolding, but its language-model bias toward grammatical correctness makes it unsuitable for verbatim learner transcription without human oversight. The hybrid pipeline addresses this by design.

**2. Non-Reproducibility is Expected** — Whisper's output varies between runs due to beam search decoding. This is acceptable because the ASR layer is a draft; the human audit is authoritative.

**3. Offset Sensitivity** — `start_offset_sec` requires per-file calibration. File `038012_EIT-2A.mp3` needed a 720s offset versus 150s for others. An adaptive VAD pre-pass could automate this.

**4. Scalability** — The manual audit bottleneck could be reduced by fine-tuning an ASR model on verified Test I data, creating a feedback loop that improves draft quality over time.

---

> ✅ **Test I Complete.** All 120 transcriptions are research-grade, preserving exact learner production including disfluencies and phonological errors. Ready for automated scoring in **Test II →**